[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/028_pytorch_nn/pytorch_nn.ipynb)

# Pytorch - Redes Neuronales

En el post [anterior](https://sensioai.com/blog/027_pytorch_intro) hicimos una introducción al framework de `redes neuronales` `Pytorch`. Hablamos de sus tres elementos fundamentales: el objeto `tensor` (similar al `array` de `NumPy`) `autograd` (que nos permite calcular derivadas de manera automáticas) y el soporte GPU. En este post vamos a entrar en detalle en la  funcionalidad que nos ofrece la librería para diseñar redes neuronales de manera flexible.

In [1]:
import torch

## Modelos secuenciales

La forma más sencilla de definir una `red neuronal` en `Pytorch` es utilizando la clase `Sequentail`. Esta clase nos permite definir una secuencia de capas, que se aplicarán de manera secuencial (las salidas de una capa serán la entrada de la siguiente). Ésto ya lo conocemos de posts anteriores, ya que es la forma ideal de definir un `Perceptrón Multicapa`.

In [2]:
D_in, H, D_out = 784, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

# D_in, H1, H2, D_out = 784, 100, 50, 10
# model = torch.nn.Sequential(
#     torch.nn.Linear(D_in, H1),
#     torch.nn.ReLU(),
#     torch.nn.Linear(H1, H2),
#     torch.nn.ReLU(),
#     torch.nn.Linear(H2, D_out),
# )


El modelo anterior es un `MLP` con 784 entradas, 100 neuronas en la capa oculta y 10 salidas. Podemos usar este modelo para hacer un clasificador de imágenes con el dataset MNIST. Pero primero, vamos a ver como podemos calcular las salidas del modelo a partir de unas entradas de ejemplo.

In [3]:
outputs = model(torch.randn(600, 784))
outputs.shape

torch.Size([600, 10])

In [4]:
print(outputs[0][:])

tensor([-0.3485,  0.1516, -0.0328, -0.0330, -0.2532,  0.0915, -0.4788,  0.1179,
        -0.3531,  0.0438], grad_fn=<SliceBackward0>)


Como puedes ver, simplemente le pasamos los inputs al modelo (llamándolo como una función). En este caso, usamos un tensor con 64 vectores de 784 valores. Es importante remarcar que los modelos de `Pytorch` (por lo general) siempre esperan que la primera dimensión sea la dimensión *batch*. Si queremos entrenar esta red en una GPU, es tan sencillo como

In [5]:
model

Sequential(
  (0): Linear(in_features=784, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=10, bias=True)
)

In [6]:
model.to("cuda")

Sequential(
  (0): Linear(in_features=784, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=10, bias=True)
)

Vamos a ver ahora como entrenar este modelo con el dataset MNIST.

In [7]:
from sklearn.datasets import fetch_openml

# descarga datos

mnist = fetch_openml('mnist_784', version=1)
X, Y = mnist["data"], mnist["target"]

X.shape, Y.shape

((70000, 784), (70000,))

In [8]:
# import numpy as np

# normalización y split
import numpy as np
x_2=np.array(X)
y_2=np.array(Y)

# normalización y split

X_train =x_2[:60000] / 255.
X_test =x_2[60000:] / 255.
y_train = y_2[:60000].astype(np.int32)
y_test = y_2[60000:].astype(np.int32)



# X_train, X_test, y_train, y_test = X[:60000] / 255., X[60000:] / 255., Y[:60000].astype(np.float32), Y[60000:].astype(np.float32)

In [9]:
# función de pérdida y derivada

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1,keepdims=True)

def cross_entropy(output, target):
    logits = output[torch.arange(len(output)), target]
    loss = - logits + torch.log(torch.sum(torch.exp(output), axis=-1))
    loss = loss.mean()
    return loss

In [10]:
# X_train

In [11]:
torch.cuda.is_available()

True

In [12]:
print(X)

       pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  pixel8  pixel9  \
0           0       0       0       0       0       0       0       0       0   
1           0       0       0       0       0       0       0       0       0   
2           0       0       0       0       0       0       0       0       0   
3           0       0       0       0       0       0       0       0       0   
4           0       0       0       0       0       0       0       0       0   
...       ...     ...     ...     ...     ...     ...     ...     ...     ...   
69995       0       0       0       0       0       0       0       0       0   
69996       0       0       0       0       0       0       0       0       0   
69997       0       0       0       0       0       0       0       0       0   
69998       0       0       0       0       0       0       0       0       0   
69999       0       0       0       0       0       0       0       0       0   

       pixel10  ...  pixel7

In [13]:
# convertimos datos a tensores y copiamos en gpu

X_t = torch.from_numpy(X_train).float().cuda()
Y_t = torch.from_numpy(y_train).long().cuda()

# bucle entrenamiento
epochs = 350
lr = 0.8
log_each = 10
l = []
for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = cross_entropy(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    model.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

Epoch 10/350 Loss 1.84345
Epoch 20/350 Loss 1.54411
Epoch 30/350 Loss 1.25935
Epoch 40/350 Loss 1.06624
Epoch 50/350 Loss 0.93365
Epoch 60/350 Loss 0.83756
Epoch 70/350 Loss 0.76184
Epoch 80/350 Loss 0.70305
Epoch 90/350 Loss 0.65905
Epoch 100/350 Loss 0.62037
Epoch 110/350 Loss 0.58658
Epoch 120/350 Loss 0.55753
Epoch 130/350 Loss 0.53224
Epoch 140/350 Loss 0.50996
Epoch 150/350 Loss 0.49014
Epoch 160/350 Loss 0.47235
Epoch 170/350 Loss 0.45627
Epoch 180/350 Loss 0.44164
Epoch 190/350 Loss 0.42825
Epoch 200/350 Loss 0.41593
Epoch 210/350 Loss 0.40456
Epoch 220/350 Loss 0.39400
Epoch 230/350 Loss 0.38418
Epoch 240/350 Loss 0.37500
Epoch 250/350 Loss 0.36639
Epoch 260/350 Loss 0.35831
Epoch 270/350 Loss 0.35069
Epoch 280/350 Loss 0.34350
Epoch 290/350 Loss 0.33669
Epoch 300/350 Loss 0.33024
Epoch 310/350 Loss 0.32411
Epoch 320/350 Loss 0.31827
Epoch 330/350 Loss 0.31271
Epoch 340/350 Loss 0.30739
Epoch 350/350 Loss 0.30232


Como puedes observar en el ejemplo, podemos calcular la salida del modelo con una simple línea. Luego calculamos la función de pérdida, y llamando a la función `backward` `Pytorch` se encarga de calcular las derivadas de la misma con respecto a todos los parámetros del modelo automáticamente (si no queremos acumular estos gradientes, nos aseguramos de llamar a la función `zero_grad` para ponerlos a cero antes de calcularlos). Por útlimo, podemos iterar por los parámetros del modelo aplicando la regla de actualización deseada (en este caso usamos `descenso por gradiente`).

In [14]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    y_pred = model(x)
    y_probas = softmax(y_pred)
    return torch.argmax(y_probas, axis=1)

y_pred = evaluate(torch.from_numpy(X_test).float().cuda())
accuracy_score(y_test, y_pred.cpu().numpy())

0.961

Existen algunos tipos de capas que se comportan diferente en función de si estamos entrenando la red o usándola para generar predicciones. Podemos controlar el modo en el que queremos que esté nuestra red con las funciones `train` y `eval`.

## Optimizadores y Funciones de pérdida

En el ejemplo anterior hemos calculado la función de pérdida y aplicado la regla de optimización de forma manual. Sin embargo, `Pytorch` nos ofrece funcionalidad que nos abstrae estos cálculos ofreciendo además flexibilidad para aplicar diferentes funciones de pérdida o algoritmos de optimización de manera sencilla. Podemos encontrar diferentes funciones de pérdida ya implementadas en el paquete `torch.nn`.

In [15]:
criterion = torch.nn.CrossEntropyLoss()

Mientras que los optimizadores se encuentran en el paquete `torch.optim`

In [16]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

Puedes ver la lista completa de funciones de pérdida y optimizadores disponibles en la [documentación](https://pytorch.org/docs/stable/index.html), aunque como ya has visto siempre puedes definir los tuyos propios fácilmente.

Una vez definidos estos dos objetos, nuestro bucle de entrenamiento se simplifica considerablemente.

In [17]:
D_in, H, D_out = 784, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 1500
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cuda())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/1500 Loss 1.73418
Epoch 20/1500 Loss 1.36056
Epoch 30/1500 Loss 1.10742
Epoch 40/1500 Loss 0.94173
Epoch 50/1500 Loss 0.83174
Epoch 60/1500 Loss 0.75400
Epoch 70/1500 Loss 0.68825
Epoch 80/1500 Loss 0.63705
Epoch 90/1500 Loss 0.59708
Epoch 100/1500 Loss 0.56365
Epoch 110/1500 Loss 0.53421
Epoch 120/1500 Loss 0.50878
Epoch 130/1500 Loss 0.48659
Epoch 140/1500 Loss 0.46700
Epoch 150/1500 Loss 0.44954
Epoch 160/1500 Loss 0.43384
Epoch 170/1500 Loss 0.41963
Epoch 180/1500 Loss 0.40667
Epoch 190/1500 Loss 0.39480
Epoch 200/1500 Loss 0.38386
Epoch 210/1500 Loss 0.37374
Epoch 220/1500 Loss 0.36433
Epoch 230/1500 Loss 0.35556
Epoch 240/1500 Loss 0.34735
Epoch 250/1500 Loss 0.33965
Epoch 260/1500 Loss 0.33240
Epoch 270/1500 Loss 0.32557
Epoch 280/1500 Loss 0.31910
Epoch 290/1500 Loss 0.31298
Epoch 300/1500 Loss 0.30717
Epoch 310/1500 Loss 0.30164
Epoch 320/1500 Loss 0.29637
Epoch 330/1500 Loss 0.29134
Epoch 340/1500 Loss 0.28654
Epoch 350/1500 Loss 0.28194
Epoch 360/1500 Loss 0.27753
E

0.9776

## Modelos custom

Si bien en muchos casos definir una `red neuronal` como una secuencia de capas es suficiente, en otros casos será un factor limitante. Un ejemplo son las redes residuales, en las que no sólo utilizamos la salida de una capa para alimentar la siguiente si no que, además, le sumamos su propia entrada. Este tipo de arquitectura no puede ser definida con la clase `Sequential`, y para ello necesitamos hacer un modelo *customizado*. Para ello, `Pytroch` nos ofrece la siguiente sintaxis.

In [18]:
# creamos una clase que hereda de `torch.nn.Module`

class ModeloPersonalizado(torch.nn.Module):

    # constructor
    def __init__(self, D_in, H, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloPersonalizado, self).__init__()

        # definimos nuestras capas
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

En primer lugar, necesitamos definir una nueva clase que herede de la clase `torch.nn.Module`. Esta clase madre aportará toda la funcionalidad esencial que necesita una `red neuronal` (soporte GPU, iterar por sus parámeteros, etc). Luego, en esta clase necesitamos definir mínimos dos funciones:

- `init`: en el constructor llamaremos al constructor de la clase madre y después definiremos todas las capas que querramos usar en la red.
- `forward`: en esta función definimos toda la lógica que aplicaremos desde que recibimos los inputs hasta que devolvemos los outputs.

En el ejemplo anterior simplemente hemos replicado la misma red (puedes conseguir el mismo efecto usando la clase `Sequential`).

In [19]:
model = ModeloPersonalizado(784, 100, 10)
# Codigo para saber si el modelo esta votando los datos en las cantidades correctas
x_prueba = torch.randn(500, 784)
print(x_prueba)
outputs = model(x_prueba)
outputs.shape

tensor([[-0.7535,  0.5868,  0.0187,  ..., -0.3015,  2.4863, -0.8185],
        [ 0.2155,  1.5786, -0.4305,  ..., -0.0231,  0.0318, -0.9933],
        [ 1.5647,  1.1627,  0.8018,  ...,  1.2959, -0.0943, -0.9678],
        ...,
        [ 0.0585,  0.3967, -0.7600,  ..., -0.4182, -0.5821,  1.1825],
        [-0.0031, -0.6819, -1.0362,  ..., -0.7461, -0.6972,  1.0222],
        [ 1.1802,  0.8016, -0.7811,  ..., -1.6133, -1.1178,  1.2393]])


torch.Size([500, 10])

Ahora, podemos entrenar nuestra red de la misma forma que lo hemos hecho anteriormente.

In [20]:
model.to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cuda())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 1.75285
Epoch 20/100 Loss 1.40834
Epoch 30/100 Loss 1.15540
Epoch 40/100 Loss 0.98892
Epoch 50/100 Loss 0.86284
Epoch 60/100 Loss 0.77381
Epoch 70/100 Loss 0.71940
Epoch 80/100 Loss 0.66624
Epoch 90/100 Loss 0.62225
Epoch 100/100 Loss 0.58575


0.9304

Aquí puedes ver otro ejemplo de como definir un `MLP` con conexiones residuales, algo que no podemos hacer simplemente usando un modelo secuencial.

In [21]:
class ModelCustom2(torch.nn.Module):

    def __init__(self, D_in, H, D_out):
        super(ModelCustom2, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)
        x = self.relu(x1)
        x = self.fc2(x + x1)
        return x

In [22]:
model = ModelCustom2(784, 100, 10).to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().cuda())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 1.60162
Epoch 20/100 Loss 1.16825
Epoch 30/100 Loss 0.95601
Epoch 40/100 Loss 0.86245
Epoch 50/100 Loss 0.77476
Epoch 60/100 Loss 0.71081
Epoch 70/100 Loss 0.66247
Epoch 80/100 Loss 0.62461
Epoch 90/100 Loss 0.59452
Epoch 100/100 Loss 0.57068


0.904

De esta manera, tenemos mucha flexibilidad para definir nuestras redes.

## Accediendo a las capas de una red

En ocasiones queremos acceder a una capa en particular de nuestra red. Para ello, podemos acceder utilizando su nombre.

In [23]:
model

ModelCustom2(
  (fc1): Linear(in_features=784, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=10, bias=True)
)

In [24]:
model.fc1

Linear(in_features=784, out_features=100, bias=True)

También podemos acceder directamente a los tensores que contienen los parámetros con las propiedades adecuadas

In [25]:
model.fc1.weight

Parameter containing:
tensor([[ 0.0118, -0.0017,  0.0336,  ..., -0.0119, -0.0271,  0.0061],
        [ 0.0283,  0.0099,  0.0291,  ..., -0.0202, -0.0119,  0.0218],
        [ 0.0343,  0.0057,  0.0323,  ...,  0.0205,  0.0169,  0.0132],
        ...,
        [ 0.0243,  0.0020, -0.0043,  ..., -0.0131,  0.0181,  0.0155],
        [-0.0298, -0.0047, -0.0180,  ..., -0.0207,  0.0141, -0.0089],
        [-0.0025,  0.0205, -0.0093,  ..., -0.0316,  0.0176,  0.0002]],
       device='cuda:0', requires_grad=True)

In [26]:
model.fc1.bias

Parameter containing:
tensor([-0.0194,  0.0092,  0.0699, -0.0186,  0.0168,  0.0111, -0.0007, -0.0169,
         0.0423, -0.0298, -0.0041, -0.0037, -0.0764,  0.0261,  0.0316,  0.0504,
         0.0711,  0.0264,  0.0103,  0.0049,  0.0193,  0.0630,  0.0355,  0.0305,
         0.0627,  0.0057, -0.0266, -0.0030, -0.0253,  0.0337,  0.0016,  0.0473,
        -0.0203, -0.0201,  0.0089,  0.0344,  0.0244,  0.0083, -0.0250, -0.0558,
         0.0178, -0.0352,  0.0246,  0.0227, -0.0055,  0.0215,  0.0317,  0.0675,
        -0.0103,  0.0095, -0.0170, -0.0062,  0.0072,  0.0031,  0.0255, -0.0012,
         0.0231,  0.0027, -0.0029,  0.0209, -0.0048, -0.0179,  0.0229, -0.0048,
         0.0120,  0.0423, -0.0565,  0.0261,  0.0225,  0.0143,  0.0235,  0.0701,
         0.0340,  0.0083, -0.0439,  0.0035,  0.0343,  0.0186,  0.0270,  0.0445,
         0.0220,  0.0064,  0.0118,  0.0112,  0.0501,  0.0119,  0.0951,  0.0357,
         0.0446,  0.0547,  0.0452, -0.0420, -0.0121,  0.0909,  0.0188,  0.0154,
         0.0521,  

Es posible sobreescribir una capa de la siguiente manera

In [27]:
model.fc2 = torch.nn.Linear(100, 1)

model

ModelCustom2(
  (fc1): Linear(in_features=784, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=1, bias=True)
)

Ahora, la capa final de nuestra red tiene solo una salida. Esta nueva capa ha sido inicializada de manera aleatoria, por lo que esta nueva red no nos va a servir de mucho. Sin embargo, podríamos volver a entrenar esta red en otro problema en el que solo necesitemos una salida aprovechando los pesos que ya hemos entrenado anteriormente con el dataset MNIST. Esto es la base del *transfer learning*, una técnica que utilizaremos muchísimo más adelante y la cual explicaremos en detalle.

A continuación encontrarás varios trucos a la hora de crear redes neuronales a partir de otras que te pueden resultar útiles.

In [28]:
# obtener una lista con las capas de una red

list(model.children())

[Linear(in_features=784, out_features=100, bias=True),
 ReLU(),
 Linear(in_features=100, out_features=1, bias=True)]

In [29]:
# crear nueva red a partir de la lista (excluyendo las útlimas dos capa)

new_model = torch.nn.Sequential(*list(model.children())[:-2])
new_model

Sequential(
  (0): Linear(in_features=784, out_features=100, bias=True)
)

In [30]:
# crear nueva red a partir de la lista (excluyendo las útlima capa)

new_model = torch.nn.ModuleList(list(model.children())[:-1])
new_model

ModuleList(
  (0): Linear(in_features=784, out_features=100, bias=True)
  (1): ReLU()
)

## Resumen

En este post hemos visto la funcionalidad que `Pytorch` nos ofrece a la hora de definir y entrenar nuestras `redes neuronales`. El paquete `torch.nn` contiene todo lo necesario para diseñar nuestros modelos, ya sea de manera secuencial o con una clase *custom* para arquitecturas más complicadas. También nos da muchas funciones de pérdida que podemos usar directamente para entrenar las redes. Te recomiendo encarecidamente que le eches un vistazo a la [documentación](https://pytorch.org/docs/stable/nn.html) par hacerte una idea de todo lo que puedes hacer. También hemos visto como el paquete `torch.optim` nos oferece algoritmos de optimización que también nos hacen la vida más fácil a la hora de entrenar nuestras redes.